# Projet Python - pollution atmosphérique et indices de mortalité et richesse 

## Introduction 

### Explications/justifications 

raconter une histoire: pitch(quel sujet, problématique, contexte (pq ce sujet), données (où, pq))

### Chargement des packages 

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import s3fs
import geopandas as gpd
np.random.seed(123) 

## Récupération des données et premières manipulations 

description (infos générales, NA, outliers, MaR, champs statistique (qui est couvert par mes données))

### a. Pollution

### b. Mortalité 

### c. Prix de l'immobilier 

Pour obtenir des données publiques sur les prix de l'immobilier en Île de France, on va utiliser le site datagouv qui met a disposition un csv contenant des prix median au m2 par commune et par année dans pas mal de communes en France.
Le seul soucis est qu'il ne couvre pas toutes les communes pour certaines raison:
- Il faut que la commune concernée possède une page sur le site Terralyze.
- Il faut qu'il y ait eu plus de 5 ventes éffectuée durant l'année.
- Il faut que le bien vendu ait une surface comprise entre 20 et 500 m2 et un prix au m2 compris entre 1000 euros et 50 000 euros pour être pris en compte.

In [3]:
url = 'https://static.data.gouv.fr/resources/prix-immobilier-par-commune-ventes-2014-a-2025-dvf-millesime-2026/20261026-112727/prix-immobilier-communes-series-2014-2025.csv'
response = requests.get(url)
if response.status_code == 200:
    with open("data_immo_price.csv", "wb") as file:
        file.write(response.content)

In [23]:
df = pd.read_csv("data_immo_price.csv")
df.head(1)

,code_insee,commune,code_departement,departement,region,population,annee,prix_m2_median,prix_m2_median_appartement,prix_m2_median_maison,prix_m2_median_neuf,ventes,ventes_maisons,ventes_appartements,source
0,01004,Ambérieu-en-Bugey,01,Ain,Auvergne-Rhône-Alpes,15934.0,2014,1840.0,1994.0,1672.0,2684.0,139,NaN,NaN,"DVF (demandes de valeurs foncières, DGFiP, Lic..."


In [24]:
print(f'Le Dataframe comporte les colonnes suivantes: {df.columns} et a pour shape: {df.shape}. En tout il comporte les données de {df['commune'].nunique()} communes.')

Le Dataframe comporte les colonnes suivantes: Index(['code_insee', 'commune', 'code_departement', 'departement', 'region',
       'population', 'annee', 'prix_m2_median', 'prix_m2_median_appartement',
       'prix_m2_median_maison', 'prix_m2_median_neuf', 'ventes',
       'ventes_maisons', 'ventes_appartements', 'source'],
      dtype='str') et a pour shape: (61327, 15). En tout il comporte les données de 5208 communes.


Nettoyons le afin de conserver un DataFrame plus propre avec une ligne par communes et une colonne par année.

In [ ]:
df = df.drop(columns = ['population', 'prix_m2_median_appartement','prix_m2_median_maison','prix_m2_median_neuf','ventes', 'ventes_maisons', 'ventes_appartements','source'])

In [30]:
#Comme le DataFrame contient une ligne par année et par commune, on le tourne pour l'indexer sur les communes et mettre les années en colonnes.
P1 = df.pivot(
    index = 'code_insee',
    columns = 'annee',
    values = 'prix_m2_median'
)
P1.head()
#On reconstruit aussi les colonnes concernant la region et le nom de la commune
P2 = df[['code_insee', 'commune', 'region']].drop_duplicates(subset = 'code_insee')
#On merge les deux dataframe intermédiaire
Prix_immo = P2.merge(
    P1,
    left_on = 'code_insee',
    right_on = 'code_insee',
    how = 'right'
)

In [31]:
Prix_immo.head()

,code_insee,commune,region,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,01004,Ambérieu-en-Bugey,Auvergne-Rhône-Alpes,1840.0,1824.0,NaN,NaN,1855.0,2068.0,1975.0,2189.0,2419.0,2322.0,2276.0,2485.0
1,01007,Ambronay,Auvergne-Rhône-Alpes,1705.0,1439.0,1990.0,1798.0,1862.0,1786.0,1644.0,2419.0,2463.0,2595.0,2510.0,2146.0
2,01014,Arbent,Auvergne-Rhône-Alpes,1825.0,1685.0,1832.0,1133.0,1700.0,1385.0,1257.0,1910.0,2107.0,1817.0,1935.0,2046.0
3,01024,Attignat,Auvergne-Rhône-Alpes,1751.0,1569.0,1705.0,1777.0,1830.0,1857.0,1911.0,2171.0,2315.0,2275.0,2400.0,2045.0
4,01025,Bâgé-Dommartin,Auvergne-Rhône-Alpes,1644.0,NaN,1742.0,1747.0,1775.0,1837.0,1886.0,2112.0,2323.0,2260.0,1981.0,2471.0


## Statistiques descriptives/Visualisation

### a. Pollution

### b. Mortalité 

### c. Prix de l'immobilier 

## Liens entre nos indices 

### a. Pollution et mortalité 

### b. Pollution et prix de l'immobilier 

## Conclusion